# Phase 2 · Train with the reward (Kaggle, 2× T4)

Fine-tunes **Qwen2.5-1.5B-Instruct** for English→German. Then it compares four ways of using the training reward chosen in Phase 1 (**MetricX-24-QE**), all from the same starting point and with the same budget of 5,000 prompts × 8 samples:

| System | What it is |
|---|---|
| `sft` | Supervised fine-tuning on ~20k News Commentary pairs; the shared starting point |
| `sft+bo8` | Best-of-8: sample 8 translations at test time, keep the one MetricX-QE scores highest |
| `rsft` | Fine-tune on the best of 8 samples per prompt |
| `dpo` | DPO on (best, worst) pairs of the same samples |
| `grpo` | Online GRPO against MetricX-QE; guard checks only *logged* |
| `grpo_guarded` | The same, with penalties for wrong language, number mismatch, length and copying (the blind spots from Phase 1) |
| `base`, `opus-mt` | Zero-shot Qwen and a classic NMT model, as anchors |

**Two parts, one session each** (set `PART` below). Rough times on 2× T4, not yet measured:
- **Part A** (~4–5 h): data, SFT, sampling and scoring, RSFT and DPO, and translations for those systems.
- **Part B** (~5–6 h): GRPO and guarded GRPO side by side (one per GPU), then their translations and checkpoint curves.

### Setup (once)
1. Kaggle: *Settings* → Accelerator **GPU T4 ×2**, Internet **On**.
2. **`HF_TOKEN` must now be a *write* token** (huggingface.co → Settings → Access Tokens → *Write*). Adapters are saved to a private repo `<your-user>/mtrlhf-en-de-adapters` as they finish, so Part B (or a re-run after a stopped session) picks them up. With a read-only token everything still runs, but adapters only live inside the session.
3. `GH_TOKEN` as in Phase 1: translations, logs and the sampled candidates are pushed to `results/p2/` on GitHub.
4. For long runs use **Save Version → Save & Run All** so you can close the tab.

Every step skips work that is already saved, so re-running the notebook continues where it stopped.

In [ ]:
# ---- configuration ----
REPO = "https://github.com/guohere/machine-translation-with-RLHF.git"
BRANCH = "claude/compassionate-ritchie-2nga86"
PART = "A"                     # "A", then "B" in a second session (or "ALL" on a fast GPU)
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # "Qwen/Qwen2.5-0.5B-Instruct" for a ~3x faster trial run
HUB_REPO = "auto"              # private HF repo for adapters; None = keep them in this session only
SMOKE_TEST = True              # run a ~10-minute tiny version of every step first
RUN_GRPO_COMETKIWI = False     # optional extra run: does GRPO on CometKiwi drift into the wrong language?
TEST_SETS = "flores_devtest.en-de wmt24.en-de flores_devtest.de-en flores_devtest.en-zh"
QUICK_SCORES = "chrf metricx24_qe comet22"   # comet22 = the held-out evaluator from Phase 1

In [ ]:
# ---- GPU check and Hugging Face token (works on Kaggle and Colab) ----
import os, subprocess

WORK = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
os.chdir(WORK)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - switch the accelerator to a T4 GPU before continuing")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

token = get_secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN loaded")
else:
    print("WARNING: no HF_TOKEN secret found - the gated CometKiwi model will fail to download")

In [ ]:
# ---- get the code and install (~3 minutes) ----
if not os.path.isdir("machine-translation-with-RLHF"):
    !git clone -q -b {BRANCH} {REPO}
else:
    !git -C machine-translation-with-RLHF pull -q
os.chdir(os.path.join(WORK, "machine-translation-with-RLHF"))
!git log -1 --format="code version: %h %s"
# transformers stays below 5 so the COMET evaluator works next to TRL (see pyproject.toml)
!pip install -q -e ".[phase1,phase2]"
!pip install -q --no-deps unbabel-comet==2.2.7 "git+https://github.com/google-research/mt-metrics-eval"
!python -c "import torch, transformers, trl, peft, comet; print('ready: torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| GPUs', torch.cuda.device_count())"
os.environ["MTRLHF_HUB_REPO"] = HUB_REPO or ""

In [ ]:
# ---- save results to GitHub as the run goes (needs GH_TOKEN) ----
import base64, gzip, shutil, subprocess

GH_TOKEN = get_secret("GH_TOKEN")
RESULTS = "results/p2"
print("GH_TOKEN loaded" if GH_TOKEN else "No GH_TOKEN: results stay in this session only")

def _git(*args):
    auth = base64.b64encode(f"x-access-token:{GH_TOKEN}".encode()).decode()
    cmd = ["git", "-c", "user.name=Phase 2 notebook", "-c", "user.email=noreply@users.noreply.github.com",
           "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}", *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).replace(GH_TOKEN, "***").replace(auth, "***")

def save_results(message):
    """Copy small artifacts into results/p2/ and push them to the branch."""
    os.makedirs(RESULTS, exist_ok=True)
    for f in ("data_stats.json", "pairs_stats.json"):
        if os.path.exists(f"data/p2/{f}"):
            shutil.copy(f"data/p2/{f}", RESULTS)
    if os.path.exists("data/p2/candidates.jsonl"):
        with open("data/p2/candidates.jsonl", "rb") as src, gzip.open(f"{RESULTS}/candidates.jsonl.gz", "wb") as dst:
            shutil.copyfileobj(src, dst)
    for run_dir in glob.glob("runs/p2/*"):
        name = os.path.basename(run_dir)
        for f in ("log_history.json", "samples.jsonl"):
            if os.path.exists(f"{run_dir}/{f}"):
                os.makedirs(f"{RESULTS}/training/{name}", exist_ok=True)
                shutil.copy(f"{run_dir}/{f}", f"{RESULTS}/training/{name}/")
    if not GH_TOKEN:
        return
    _git("add", RESULTS)
    if _git("diff", "--cached", "--quiet")[0] != 0:
        code, out = _git("commit", "-q", "-m", message)
        if code:
            print("commit failed:", out); return
    code, out = _git("pull", "-q", "--rebase", "origin", BRANCH)
    if code:
        print("pull failed:", out); return
    code, out = _git("push", "-q", "origin", f"HEAD:{BRANCH}")
    print(f"saved to GitHub: {message}" if code == 0 else f"push failed: {out}")

def restore_candidates():
    if os.path.exists(f"{RESULTS}/candidates.jsonl.gz") and not os.path.exists("data/p2/candidates.jsonl"):
        os.makedirs("data/p2", exist_ok=True)
        with gzip.open(f"{RESULTS}/candidates.jsonl.gz", "rb") as src, open("data/p2/candidates.jsonl", "wb") as dst:
            shutil.copyfileobj(src, dst)
        print("restored sampled candidates from an earlier run")

import glob, sys
sys.path.insert(0, "src")
from mtrlhf.models import CheckpointStore
store = CheckpointStore.from_env("ckpt", HUB_REPO)
print("adapter store:", store.hub_repo or "local only (this session)")

import time

def n_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    except FileNotFoundError:
        return 0
    return sum(line.startswith("GPU") for line in out.splitlines())

def run_jobs(jobs, report_every=600):
    """Run (name, shell command) jobs, two at a time on two GPUs if available, else one after another.
    Output goes to <name>.log; the last log line is printed every 10 minutes."""
    slots = max(1, min(n_gpus(), 2))
    pending, running, last = list(jobs), [], time.time()
    while pending or running:
        while pending and len(running) < slots:
            name, cmd = pending.pop(0)
            gpu = min(set(range(slots)) - {r[2] for r in running})
            log = open(f"{name}.log", "w")
            proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT,
                                    env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
            running.append((name, proc, gpu, log))
            print(f"started {name} on GPU {gpu}", flush=True)
        time.sleep(15)
        for r in list(running):
            if r[1].poll() is not None:
                r[3].close()
                running.remove(r)
                tail = open(f"{r[0]}.log").read().strip().splitlines()[-3:]
                print(f"{r[0]} finished (exit code {r[1].returncode}):", *tail, sep="\n  ", flush=True)
        if running and time.time() - last > report_every:
            last = time.time()
            for name, *_ in running:
                lines = open(f"{name}.log").read().strip().splitlines()
                print(f"[{time.strftime('%H:%M')}] {name}: {lines[-1][-160:] if lines else '(starting)'}", flush=True)

### Step 1 · Data and test sets (CPU, ~5–10 min)
News Commentary en-de → 20k SFT pairs, 500 validation pairs, and 5,000 English prompts (disjoint). FLORES-200 dev/devtest (en-de, de-en, en-zh) and WMT24 en-de. Training sentences that occur in any test set are removed.

In [ ]:
if not os.path.exists("data/p2/prompts.jsonl"):
    !python scripts/p2_prepare_data.py --out data/p2
!cat data/p2/data_stats.json
save_results("Phase 2: data statistics")

### Smoke test (~10 min, first time only)
A tiny version of SFT → sampling → GRPO with the 0.5B model and the real MetricX reward. It checks GPU memory, logins and library versions before the long runs. It writes to `ckpt_debug/` and `runs/p2_debug/` and does not touch the real results.

In [ ]:
if SMOKE_TEST:
    !python scripts/p2_train.py --config configs/p2/sft.yaml --debug
    !python scripts/p2_sample.py --system sft --ckpt ckpt_debug --limit 8 --n 4 --out runs/p2_debug/candidates.jsonl
    !python scripts/p2_train.py --config configs/p2/grpo_guarded.yaml --debug
    !python scripts/p2_translate.py --systems sft --ckpt ckpt_debug --testsets flores_devtest.en-de --limit 8 --out runs/p2_debug/outputs --score chrf

### Part A · Step 2 · Supervised fine-tuning → `sft` (~45–60 min)

In [ ]:
if PART in ("A", "ALL"):
    !python scripts/p2_train.py --config configs/p2/sft.yaml --set base_model={BASE_MODEL}
    save_results("Phase 2: sft")

### Part A · Step 3 · Sample 8 translations per prompt and score them with MetricX-QE (~60–75 min)
This is the shared budget: 5,000 prompts × 8 samples = 40,000 reward calls. The policy runs on GPU 0 and MetricX on GPU 1. Resumable.

In [ ]:
if PART in ("A", "ALL"):
    restore_candidates()
    !python scripts/p2_sample.py --system sft --n 8 --reward metricx24_qe --out data/p2/candidates.jsonl
    !python scripts/p2_build_pairs.py --candidates data/p2/candidates.jsonl --out data/p2
    save_results("Phase 2: sampled candidates and pairs")

### Part A · Step 4 · RSFT and DPO (~30–45 min; side by side on two GPUs)

In [ ]:
if PART in ("A", "ALL"):
    run_jobs([("rsft", "python scripts/p2_train.py --config configs/p2/rsft.yaml"),
              ("dpo", "python scripts/p2_train.py --config configs/p2/dpo.yaml")])
    save_results("Phase 2: rsft and dpo")

### Part B · Step 5 · GRPO and guarded GRPO (~4–5 h side by side on two GPUs; twice as long on one)
Both start from `sft`. Every 125 steps the adapter is saved as `<name>-step<k>`. Step 625 (= 5,000 prompts × 8 samples) is the **matched-budget** model used in the comparison. Training continues to step 1250 only to trace the over-optimization curve. If the session stops after step 625, the main result is already saved, and the run is skipped next time.

Watch the training log: `rewards/metricx24_qe/mean` should rise. The `guard_*` lines are the exploit rates (logged in both runs, penalized only in `grpo_guarded`).

In [ ]:
BUDGET_STEP = 625

def grpo_job(name):
    if store.has(name) or store.has(f"{name}-step{BUDGET_STEP}"):
        print(f"{name}: matched-budget checkpoint already saved, skipping")
        return []
    return [(name, f"python scripts/p2_train.py --config configs/p2/{name}.yaml --force")]

if PART in ("B", "ALL"):
    runs = ["grpo", "grpo_guarded"] + (["grpo_cometkiwi"] if RUN_GRPO_COMETKIWI else [])
    run_jobs([job for name in runs for job in grpo_job(name)])
    save_results("Phase 2: GRPO runs")

### Step 6 · Translate the test sets (~1 h per part)
Greedy decoding for every system, plus best-of-8 for `sft`. In Part B, also every saved GRPO checkpoint on FLORES dev (the over-optimization curve). Results go straight to `results/p2/outputs/` and are pushed after each block.

In [ ]:
def have(name):
    return name in ("base", "opus-mt") or store.has(name)

if PART in ("A", "ALL"):
    systems = " ".join(s for s in ["base", "opus-mt", "sft", "rsft", "dpo"] if have(s))
    !python scripts/p2_translate.py --systems {systems} --testsets {TEST_SETS} flores_dev.en-de --base-model {BASE_MODEL}
    save_results("Phase 2: translations (base, opus-mt, sft, rsft, dpo)")
    !python scripts/p2_translate.py --systems sft --best-of 8 --testsets flores_devtest.en-de wmt24.en-de flores_dev.en-de
    save_results("Phase 2: translations (sft + best-of-8)")

if PART in ("B", "ALL"):
    for run in ["grpo", "grpo_guarded"] + (["grpo_cometkiwi"] if RUN_GRPO_COMETKIWI else []):
        main = f"{run}-step{BUDGET_STEP}"
        if have(main):
            !python scripts/p2_translate.py --systems {main} --testsets {TEST_SETS} flores_dev.en-de
        curve = " ".join(f"{run}-step{k}" for k in range(125, 1251, 125) if have(f"{run}-step{k}"))
        if curve:
            !python scripts/p2_translate.py --systems {curve} --testsets flores_dev.en-de
        save_results(f"Phase 2: translations ({run})")

### Step 7 · Quick evaluation (~15–30 min)
Mean chrF, MetricX-QE (the training reward) and COMET-22 (held out) per system and test set, plus how often each guard fires. This is a first look; Phase 3 adds significance tests, the over-optimization plots and the hacking analysis.

In [ ]:
!python scripts/p2_translate.py --summary-only --score {QUICK_SCORES} > /dev/null
save_results(f"Phase 2: quick evaluation (part {PART})")
from IPython.display import Markdown, display
display(Markdown(open("results/p2/outputs/quick_eval.md").read()))

### Troubleshooting
| Symptom | Fix |
|---|---|
| `Hub mirror disabled` | `HF_TOKEN` is read-only. Create a *write* token, or run Part A and B in the same session |
| `CUDA out of memory` in GRPO | Add `--set batch_size=4 grad_accum=16` to the GRPO command (same 8 prompts per step), or use `BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"` |
| `CUDA out of memory` in sampling | Add `--batch-size 32` to the `p2_sample.py` command |
| A step was interrupted | Run the notebook again: finished adapters, candidates and translations are skipped |
| `push failed` | Check the `GH_TOKEN` secret (fine-grained, this repository, *Contents: Read and write*) |
| FLORES download fails | The notebook falls back to the gated `openlanguagedata/flores_plus`; accept its terms on Hugging Face |